# GUNDAM interface demo

This notebook configures GUNDAM through `gundam-interface`, initializes the fitter engine, evaluates the nominal likelihood, runs a minimization, and inspects the best-fit parameters.


In [1]:
from pathlib import Path

# User configuration
repoRoot = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
outputDirectory = repoRoot / "output" / "demo"
outputDirectory.mkdir(parents=True, exist_ok=True)

nCpuThreads = 3
pythonPath = "/Users/nadrino/Documents/Work/Install/gundam/lib"
workDir = "/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024"
configPath = "configOA2024.yaml"
overrideList = [
    "override/v12ProdRun45.yaml",
    "override/onlyFlux5.yaml",
    "override/noEigen.yaml",
]
dataType = "Asimov"  # "Asimov", "Toy", or "RealData"
seed = 12345

initializeLogPath = outputDirectory / "gundam_initialize.log"
minimizeLogPath = outputDirectory / "gundam_minimize.log"

outputDirectory


PosixPath('/Users/nadrino/Documents/Work/Repositories/gundam-python-interface/output/demo')

In [2]:
import sys

import numpy as np

# Prefer the local checkout when running this notebook before pip installation.
srcPath = repoRoot / "src"
if srcPath.exists() and str(srcPath) not in sys.path:
    sys.path.insert(0, str(srcPath))

from gundam_interface import GundamContext, GundamInterface


In [3]:
np.random.seed(seed)

context = GundamContext(
    nCpuThreads=nCpuThreads,
    pythonPath=pythonPath,
    workDir=workDir,
    configPath=configPath,
    overrideList=overrideList,
    dataType=dataType,
    randomSeed=seed,
)

context.toDict(includeConfigJsonString=False)


{'nCpuThreads': 3,
 'pythonPath': '/Users/nadrino/Documents/Work/Install/gundam/lib',
 'workDir': '/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024',
 'dataType': 'Asimov',
 'randomSeed': 12345,
 'configPath': 'configOA2024.yaml',
 'overrideList': ['override/v12ProdRun45.yaml',
  'override/onlyFlux5.yaml',
  'override/noEigen.yaml']}

In [4]:
gundam = GundamInterface(context)
gundam.configure()
gundam.initialize(logPath=initializeLogPath)

print(f"Initialized GUNDAM with {len(gundam.parameters)} active parameters")
print(f"Initialization log: {initializeLogPath}")
for index, parameter in enumerate(gundam.parameters):
    print(
        f"{index:3d}: {parameter.name} "
        f"prior={parameter.prior:.8g} "
        f"step={parameter.stepSize:.8g} "
        f"value={parameter.value:.8g}"
    )


2026.06.30 11:48:23  INFO ConfigUtils: Reading config file: /Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024/configOA2024.yaml
2026.06.30 11:48:23  INFO ConfigUtils: Overriding config with "/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024/override/v12ProdRun45.yaml"
2026.06.30 11:48:23  WARN ConfigUtils:   Added: fitterEngineConfig/propagatorConfig/dataSetList/0(name:"ND280")/mc/filePathList
2026.06.30 11:48:23  WARN ConfigUtils:   Added: fitterEngineConfig/propagatorConfig/dataSetList/0(name:"ND280")/data/0(name:"data")/filePathList
2026.06.30 11:48:23  INFO ConfigUtils: Overriding config with "/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024/override/onlyFlux5.yaml"
2026.06.30 11:48:23  WARN ConfigUtils:   Override: fitterEngineConfig/propagatorConfig/parameterSetListConfig/0(name:"Flux Systematics")/isEnabled: true -> false
2026.06.30 11:48:23  WARN ConfigUtils:   Override: fitterEngineConfig/propagatorConfig/parameterSe

In [5]:
nominalNormalizedValues = np.zeros(len(gundam.parameters), dtype=np.float64)
nominalPhysicalValues = gundam.normalizedToPhysical(nominalNormalizedValues)
nominalLlh = gundam.evaluateLlh(normalizedValues=nominalNormalizedValues)

print(f"Nominal LLH: {nominalLlh:.8g}")
print("Nominal parameters:")
for name, physicalValue in zip(gundam.parameterNames, nominalPhysicalValues):
    print(f"  - {name}: physical={physicalValue:.8g}")


Nominal LLH: -4.0107901e-14
Nominal parameters:
  - Flux Systematics/#0: physical=1
  - Flux Systematics/#1: physical=1
  - Flux Systematics/#2: physical=1
  - Flux Systematics/#3: physical=1
  - Flux Systematics/#4: physical=1


In [6]:
bestFitLlh = gundam.minimize(logPath=minimizeLogPath)
bestFitPhysicalValues = gundam.getParameterValues()
bestFitNormalizedValues = gundam.physicalToNormalized(bestFitPhysicalValues)
deltaLlh = bestFitLlh - nominalLlh

print(f"Best-fit LLH: {bestFitLlh:.8g}")
print(f"Delta LLH relative to nominal: {deltaLlh:.8g}")
print(f"Minimizer log: {minimizeLogPath}")



2026.06.30 11:48:44  INFO MinimizerBase: ──────────────────────────────
2026.06.30 11:48:44  INFO MinimizerBase: Summary of the fit parameters:
2026.06.30 11:48:44  INFO MinimizerBase: ──────────────────────────────
2026.06.30 11:48:44  INFO MinimizerBase: Flux Systematics: 100 parameters
┌───────┬──────────┬──────────┬──────────┬──────────────┬────────────────┐
│ Title │ Starting │    Prior │   StdDev │       Limits │         Status │
├───────┼──────────┼──────────┼──────────┼──────────────┼────────────────┤
│    #0 │ 1.000000 │ 1.000000 │ 0.059244 │ [-inf, +inf] │ Gaussian Prior │
│    #1 │ 1.000000 │ 1.000000 │ 0.052535 │ [-inf, +inf] │ Gaussian Prior │
│    #2 │ 1.000000 │ 1.000000 │ 0.052934 │ [-inf, +inf] │ Gaussian Prior │
│    #3 │ 1.000000 │ 1.000000 │ 0.051455 │ [-inf, +inf] │ Gaussian Prior │
│    #4 │ 1.000000 │ 1.000000 │ 0.073820 │ [-inf, +inf] │ Gaussian Prior │
└───────┴──────────┴──────────┴──────────┴──────────────┴────────────────┘
2026.06.30 11:48:44  INFO Minimize

In [7]:
print("Best-fit parameters:")
for name, physicalValue, normalizedValue in zip(
    gundam.parameterNames,
    bestFitPhysicalValues,
    bestFitNormalizedValues,
):
    print(
        f"  - {name}: "
        f"physical={physicalValue:.8g}, "
        f"normalized={normalizedValue:.8g}"
    )


Best-fit parameters:
  - Flux Systematics/#0: physical=1, normalized=1.7774809e-09
  - Flux Systematics/#1: physical=1, normalized=2.1898404e-09
  - Flux Systematics/#2: physical=1, normalized=-9.8421782e-10
  - Flux Systematics/#3: physical=1, normalized=6.0967337e-09
  - Flux Systematics/#4: physical=1, normalized=8.4696892e-08


In [8]:
# Re-evaluate the likelihood at the best-fit point as a consistency check.
reevaluatedBestFitLlh = gundam.evaluateLlh(physicalValues=bestFitPhysicalValues)

print(f"Minimizer best-fit LLH: {bestFitLlh:.8g}")
print(f"Re-evaluated best-fit LLH: {reevaluatedBestFitLlh:.8g}")
print(f"Absolute difference: {abs(reevaluatedBestFitLlh - bestFitLlh):.8g}")


Minimizer best-fit LLH: -6.0008985e-13
Re-evaluated best-fit LLH: -6.0008985e-13
Absolute difference: 0
